[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/15b_production_frameworks.ipynb)

# Production frameworks — TRL, Unsloth, Axolotl, LLaMA-Factory

> **Fine-tuning series — 15b of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. Conceptual/reference — no GPU setup needed.


## Production frameworks — TRL · Unsloth · Axolotl · LLaMA-Factory

This course teaches the mechanics using **TRL + PEFT** directly. For real training runs most people
drive one of four frameworks. By 2026 they've converged — all do LoRA/QLoRA/full/DPO/GRPO/etc. — so
the real difference is **workflow**: Python code, a speed wrapper, or YAML + a CLI.

| Framework | Interface | Reach for it when |
|---|---|---|
| **TRL** | Python | you want full control / custom loops — it's the HF stack the others build on |
| **Unsloth** | Python (drop-in) | single NVIDIA GPU, maximum speed / least VRAM (Colab, consumer cards) |
| **Axolotl** | YAML + CLI | config-driven, reproducible, multi-GPU / multi-node (FSDP / DeepSpeed) |
| **LLaMA-Factory** | YAML + CLI + Web UI | widest model/VLM coverage, no-code GUI (LlamaBoard) |

They're complementary, not rivals: Unsloth's kernels run *under* TRL's trainers, and both Axolotl
and LLaMA-Factory can call Unsloth as a backend.

### 1. TRL — the Python library (what this course uses)

You write Python; `SFTTrainer` / `DPOTrainer` / `GRPOTrainer` wrap the loop. Most control, most code.

In [ ]:
# pip install trl peft datasets
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ds = Dataset.from_list([                       # your {text} dataset (see notebook 02c)
    {"text": "<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\nHello!<|im_end|>"}])
trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct", train_dataset=ds,
    args=SFTConfig(output_dir="out", dataset_text_field="text",
                   per_device_train_batch_size=2, num_train_epochs=1, learning_rate=2e-4))
trainer.train()

### 2. Unsloth — same code, faster

A drop-in that patches the model for 2–5× speed and big VRAM savings on a **single NVIDIA GPU**
(open-source version is single-GPU). You still train with TRL's trainers — only the model load
changes.

In [ ]:
# pip install unsloth
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-0.5B-Instruct", max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, target_modules=["q_proj","k_proj","v_proj","o_proj"],
    use_gradient_checkpointing="unsloth")
# then hand `model` + `tokenizer` to TRL's SFTTrainer exactly like the TRL cell above

### 3. Axolotl — YAML + one command (no Python)

Define the whole run in one YAML file, launch with one command. Reproducible and multi-GPU-ready.
A minimal LoRA config:

```yaml
# lora.yml
base_model: NousResearch/Llama-3.2-1B
load_in_8bit: true            # -> load_in_4bit: true (with adapter: qlora) for QLoRA
adapter: lora                 # remove these two lines for full fine-tuning
lora_r: 16
lora_alpha: 32
lora_target_linear: true      # LoRA on all linear layers
datasets:
  - path: tatsu-lab/alpaca    # or a local file: my_data.jsonl
    type: alpaca              # alpaca | sharegpt | chat_template | completion
val_set_size: 0.05
sequence_len: 2048
micro_batch_size: 2
gradient_accumulation_steps: 4
num_epochs: 3
learning_rate: 0.0002
output_dir: ./outputs/lora-out
```

Then run it:

In [ ]:
!pip install axolotl
!axolotl fetch examples        # optional: grab ready-made configs to copy
!axolotl preprocess lora.yml   # validate data formatting before a long run
!axolotl train lora.yml        # train  (multi-GPU: append --launcher accelerate)
!axolotl merge-lora lora.yml   # merge the adapter into the base for serving

### 4. LLaMA-Factory — YAML + CLI + no-code Web UI

The broadest model/VLM coverage, plus a Gradio GUI (LlamaBoard). Same YAML-and-CLI idea, different
keys; one config switches between SFT / DPO / ORPO / SimPO / KTO / PPO / reward / continued
pre-training via `stage`.

```yaml
# qwen_lora_sft.yaml
model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
stage: sft                    # sft | dpo | kto | ppo | rm | pt
do_train: true
finetuning_type: lora         # lora | full | freeze
lora_rank: 8
lora_target: all
# quantization_bit: 4         # uncomment for QLoRA
dataset: alpaca_en_demo       # registered in data/dataset_info.json
template: qwen
cutoff_len: 2048
output_dir: saves/qwen-0.5b/lora/sft
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
```

Then run it:

In [ ]:
!pip install llamafactory
!llamafactory-cli train  qwen_lora_sft.yaml     # train
!llamafactory-cli export qwen_lora_sft.yaml     # merge adapter -> full model
!llamafactory-cli webui                         # launch the LlamaBoard no-code GUI

### Which one?

Start with **Unsloth notebooks** if you're on one GPU and want speed (it's what the Colab quickstarts
use). Move to **Axolotl** or **LLaMA-Factory** when you want runs defined as version-controlled YAML
or need multi-node. Stay in raw **TRL** when you need a custom training loop or a method the others
don't expose yet — which is exactly what the rest of this course shows you how to do.